# Model comparison: deep learning models

LSTM, GRU, CNN-LSTM, TD-CNN-LSTM, DeepTCN and ALSTM-FCN on the frame sequences of all 40 eye and head
features under cross-user mixing with stratified 10-fold cross-validation.
The models were trained on a GPU (Google Colab).

Input: `../data/frame_features.csv`. Output: per-fold and averaged results in `results/`.

In [ ]:
import os
import random
import time
from collections import OrderedDict

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

os.environ["PYTHONHASHSEED"] = "0"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras import callbacks, layers, models, optimizers, utils

try:
    tf.config.optimizer.set_jit(True)
except Exception:
    pass
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
except Exception:
    pass

DATA_PATH = os.path.join("..", "data", "frame_features.csv")
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

NUM_CLASSES = 4
TIMESTEPS = 60
STEP = 60
INCLUDE_LAST_PARTIAL = True
MIN_TAIL_RATIO = 0.5
TD_SUBSEQUENCES = 4
TD_SUBSEQ_LEN = 15

LR = 1e-3
EPOCHS = 200
BATCH_SZ = 256
PATIENCE = 30

df = pd.read_csv(DATA_PATH)
df = df.dropna(how="all")
df = df.dropna(subset=["window_marker"])
df["datetime"] = pd.to_datetime(df["datetime"])

EXP_FEATURE_GROUPS = OrderedDict({
    "Eye+Head (All_40)": [
        "Convergence_distance",
        "Left_Eye_Openness", "Right_Eye_Openness",
        "LeftPupilDiameter", "RightPupilDiameter",
        "LeftPupilPosInSensorX", "LeftPupilPosInSensorY", "RightPupilPosInSensorX", "RightPupilPosInSensorY",
        "NrmSRLeftEyeGazeDirX", "NrmSRLeftEyeGazeDirY", "NrmSRLeftEyeGazeDirZ",
        "NrmSRRightEyeGazeDirX", "NrmSRRightEyeGazeDirY", "NrmSRRightEyeGazeDirZ",
        "GazeOriginLclSpc_X", "GazeOriginLclSpc_Y", "GazeOriginLclSpc_Z",
        "GazeOriginWrldSpc_X", "GazeOriginWrldSpc_Y", "GazeOriginWrldSpc_Z",
        "GazeDirectionLclSpc_X", "GazeDirectionLclSpc_Y", "GazeDirectionLclSpc_Z",
        "GazeDirectionWrldSpc_X", "GazeDirectionWrldSpc_Y", "GazeDirectionWrldSpc_Z",
        "NrmLeftEyeOriginX", "NrmLeftEyeOriginY", "NrmLeftEyeOriginZ",
        "NrmRightEyeOriginX", "NrmRightEyeOriginY", "NrmRightEyeOriginZ",
        "HeadQRotationX", "HeadQRotationY", "HeadQRotationZ", "HeadQRotationW",
        "HeadEulX", "HeadEulY", "HeadEulZ",
    ],
})

## Sequence windows
Each 11-s segment (`window_marker`) is cut into 60-frame windows.

In [ ]:
def build_sequence_windows(df_src, feat_cols, timesteps=TIMESTEPS, step=STEP):
    df_local = df_src.sort_values(["window_marker", "datetime"])
    X_list, y_list, g_list = [], [], []
    for seg_id, seg_df in df_local.groupby("window_marker"):
        n = len(seg_df)
        if n == 0:
            continue
        vals_all = seg_df[feat_cols].values.astype(np.float32)
        labels = seg_df["cs"].values.astype(int)

        last_end = 0
        for start in range(0, max(0, n - timesteps + 1), step):
            end = start + timesteps
            if end > n:
                break
            cs_window = labels[start:end]
            vals, cnt = np.unique(cs_window, return_counts=True)
            X_list.append(vals_all[start:end, :])
            y_list.append(int(np.max(vals[cnt == cnt.max()])))
            g_list.append(seg_id)
            last_end = end

        if INCLUDE_LAST_PARTIAL and last_end < n:
            remain = n - last_end
            if remain >= int(np.ceil(MIN_TAIL_RATIO * timesteps)):
                tail = vals_all[last_end:, :]
                pad_len = timesteps - remain
                if pad_len > 0:
                    X_seq = np.concatenate([tail, np.repeat(tail[-1:, :], pad_len, axis=0)], axis=0)
                else:
                    X_seq = tail[:timesteps, :]
                cs_window = labels[last_end:]
                vals, cnt = np.unique(cs_window, return_counts=True)
                X_list.append(X_seq)
                y_list.append(int(np.max(vals[cnt == cnt.max()])))
                g_list.append(seg_id)

    X = np.stack(X_list, axis=0) if X_list else np.empty((0, timesteps, len(feat_cols)), dtype=np.float32)
    return X, np.array(y_list, dtype=int), np.array(g_list)


def identity_preprocess_input(X, model_name=None):
    return X


def reshape_for_td_cnn_lstm(X, model_name="TD-CNN-LSTM"):
    """Reshape (N, 60, F) into (N, 4, 15, F) sub-sequences for the time-distributed CNN."""
    n_samples, timesteps, n_features = X.shape
    return X.reshape(n_samples, TD_SUBSEQUENCES, TD_SUBSEQ_LEN, n_features).astype(np.float32)

## Models
LSTM, GRU and CNN-LSTM follow Kundu et al. (2025), TD-CNN-LSTM follows Islam et al. (2021),
DeepTCN follows Tasnim et al. (2024), and ALSTM-FCN follows Shimada et al. (2023).

In [ ]:
def compile_model(inp, out, name):
    model = models.Model(inp, out, name=name)
    model.compile(optimizer=optimizers.Adam(LR), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def build_lstm_model(input_shape, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)
    x = inp
    for _ in range(5):
        x = layers.LSTM(128, return_sequences=True, recurrent_dropout=0.15)(x)
    x = layers.LSTM(128, return_sequences=False, recurrent_dropout=0.15)(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return compile_model(inp, out, "LSTMx6_128")


def build_gru_model(input_shape, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)
    x = layers.GRU(32, return_sequences=True)(inp)
    x = layers.Dropout(0.20)(x)
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.Dropout(0.20)(x)
    x = layers.GRU(128, return_sequences=False)(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return compile_model(inp, out, "GRU_32_64_128")


def build_cnn_lstm_model(input_shape, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, activation="relu", padding="same")(inp)
    x = layers.Conv1D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.LSTM(128, return_sequences=False)(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return compile_model(inp, out, "CNN_LSTM")


def build_td_cnn_lstm_model(input_shape, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)
    x = layers.TimeDistributed(layers.Conv1D(filters=60, kernel_size=4, activation="relu"))(inp)
    x = layers.TimeDistributed(layers.Conv1D(filters=120, kernel_size=4, activation="relu"))(x)
    x = layers.TimeDistributed(layers.MaxPooling1D(pool_size=2, strides=2))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.LSTM(120, recurrent_dropout=0.2)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return compile_model(inp, out, "TD_CNN_LSTM")


def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout_rate=0.2):
    residual = x
    y = layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.SpatialDropout1D(dropout_rate)(y)
    y = layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation_rate)(y)
    y = layers.BatchNormalization()(y)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1, padding="same")(residual)
    y = layers.Add()([residual, y])
    return layers.Activation("relu")(y)


def build_deeptcn_model(input_shape, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)
    x = inp
    for dilation in [1, 2, 4]:
        x = tcn_residual_block(x, filters=64, kernel_size=3, dilation_rate=dilation, dropout_rate=0.2)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return compile_model(inp, out, "DeepTCN")


def build_alstm_fcn_model(input_shape, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)
    x_lstm = layers.LSTM(128, return_sequences=True, recurrent_dropout=0.15)(inp)
    attn_scores = layers.Dense(1)(x_lstm)
    attn_weights = layers.Softmax(axis=1)(attn_scores)
    x_attn = layers.Multiply()([x_lstm, attn_weights])
    x_attn = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x_attn)

    x_fcn = inp
    for filters, kernel in [(128, 8), (256, 5), (128, 3)]:
        x_fcn = layers.Conv1D(filters, kernel, padding="same")(x_fcn)
        x_fcn = layers.BatchNormalization()(x_fcn)
        x_fcn = layers.Activation("relu")(x_fcn)
    x_fcn = layers.GlobalAveragePooling1D()(x_fcn)

    x = layers.Concatenate()([x_attn, x_fcn])
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return compile_model(inp, out, "ALSTM_FCN")


MODELS = [
    {"name": "LSTM", "builder": build_lstm_model},
    {"name": "GRU", "builder": build_gru_model},
    {"name": "CNN-LSTM", "builder": build_cnn_lstm_model},
    {"name": "TD-CNN-LSTM", "builder": build_td_cnn_lstm_model, "preprocess": reshape_for_td_cnn_lstm},
    {"name": "DeepTCN", "builder": build_deeptcn_model},
    {"name": "ALSTM-FCN", "builder": build_alstm_fcn_model},
]

## Standardization and oversampling (training folds only)

In [ ]:
def fit_scaler_and_transform(X_tr, X_te):
    N_tr, T, F = X_tr.shape
    scaler = StandardScaler()
    scaler.fit(X_tr.reshape(-1, F))
    Xtr_scaled = scaler.transform(X_tr.reshape(-1, F)).reshape(N_tr, T, F)
    Xte_scaled = scaler.transform(X_te.reshape(-1, F)).reshape(X_te.shape[0], T, F)
    return Xtr_scaled.astype(np.float32), Xte_scaled.astype(np.float32)


def oversample_to_max(X, y, seed=SEED):
    """Random oversampling of whole sequences to the size of the majority class."""
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    max_n = counts.max()
    idx_all = []
    for c in classes:
        idx_c = np.where(y == c)[0]
        need = max_n - len(idx_c)
        idx_all.append(np.concatenate([idx_c, rng.choice(idx_c, size=need, replace=True)]) if need > 0 else idx_c)
    idx_all = np.concatenate(idx_all)
    rng.shuffle(idx_all)
    return X[idx_all], y[idx_all]

## Stratified 10-fold cross-validation (cross-user mixing)

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
ALL_RESULTS_FOLDS, ALL_RESULTS_AVG = [], []
labels_4 = [0, 1, 2, 3]

for exp_name, fea_list in EXP_FEATURE_GROUPS.items():
    feature_cols = [c for c in fea_list if c in df.columns and np.issubdtype(df[c].dtype, np.number)]

    X_all, y_all, _ = build_sequence_windows(df, feature_cols, TIMESTEPS, STEP)

    for model_cfg in MODELS:
        model_name = model_cfg["name"]
        builder = model_cfg["builder"]
        preprocess = model_cfg.get("preprocess", identity_preprocess_input)
        fold_acc, fold_prec, fold_rec = [], [], []
        fold_train_times, fold_test_times = [], []
        fold_f1_each_list, fold_f1_macro_list = [], []

        for fold_idx, (tr_idx, te_idx) in enumerate(cv.split(X_all, y_all), 1):
            X_tr_bal, y_tr_bal = oversample_to_max(X_all[tr_idx], y_all[tr_idx], seed=SEED + fold_idx)
            y_te = y_all[te_idx]
            X_tr, X_te = fit_scaler_and_transform(X_tr_bal, X_all[te_idx])
            X_tr_model = preprocess(X_tr, model_name=model_name)
            X_te_model = preprocess(X_te, model_name=model_name)
            ytr_oh = utils.to_categorical(y_tr_bal, num_classes=NUM_CLASSES)

            model = builder(input_shape=X_tr_model.shape[1:])
            es = callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=0)
            t_train0 = time.time()
            model.fit(X_tr_model, ytr_oh, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH_SZ,
                      callbacks=[es], class_weight=None, verbose=0)
            train_time_sec = time.time() - t_train0
            fold_train_times.append(train_time_sec)

            # Warm-up prediction, then the median of five timed predictions over the test fold.
            _ = model.predict(X_te_model[:min(len(X_te_model), BATCH_SZ)], batch_size=BATCH_SZ, verbose=0)
            try:
                tf.experimental.sync_devices()
            except Exception:
                pass
            times = []
            for _ in range(5):
                t0 = time.time()
                y_proba = model.predict(X_te_model, batch_size=BATCH_SZ, verbose=0)
                try:
                    tf.experimental.sync_devices()
                except Exception:
                    pass
                times.append(time.time() - t0)
            infer_time_per_sample_sec = float(np.median(times)) / len(X_te_model)
            fold_test_times.append(infer_time_per_sample_sec)
            y_pred = np.argmax(y_proba, axis=1)

            acc = accuracy_score(y_te, y_pred)
            prec = precision_score(y_te, y_pred, labels=labels_4, average=None, zero_division=0)
            rec = recall_score(y_te, y_pred, labels=labels_4, average=None, zero_division=0)
            f1_each = f1_score(y_te, y_pred, labels=labels_4, average=None, zero_division=0)
            f1_macro = f1_score(y_te, y_pred, labels=labels_4, average="macro", zero_division=0)
            fold_acc.append(acc)
            fold_prec.append(prec)
            fold_rec.append(rec)
            fold_f1_each_list.append(f1_each)
            fold_f1_macro_list.append(f1_macro)

            ALL_RESULTS_FOLDS.append({
                "experiment": exp_name, "model": model_name, "fold": fold_idx,
                "accuracy": round(acc, 4),
                **{f"precision_{i}": round(prec[i], 4) for i in range(4)},
                **{f"recall_{i}": round(rec[i], 4) for i in range(4)},
                **{f"f1_{i}": round(f1_each[i], 4) for i in range(4)},
                "f1_macro": round(f1_macro, 4),
                "balancing": "oversample",
                "train_time_sec": round(train_time_sec, 4),
                "inference_time_per_sample_ms": round(infer_time_per_sample_sec * 1000.0, 4),
            })

        avg_prec = np.mean(fold_prec, axis=0)
        avg_rec = np.mean(fold_rec, axis=0)
        avg_f1_each = np.mean(np.vstack(fold_f1_each_list), axis=0)
        ALL_RESULTS_AVG.append({
            "experiment": exp_name, "model": model_name, "balancing": "oversample",
            "test accuracy": round(float(np.mean(fold_acc)), 4),
            **{f"test precision {i}": round(avg_prec[i], 4) for i in range(4)},
            **{f"test recall {i}": round(avg_rec[i], 4) for i in range(4)},
            **{f"test f1 {i}": round(float(avg_f1_each[i]), 4) for i in range(4)},
            "test f1 macro": round(float(np.mean(fold_f1_macro_list)), 4),
            "time train mean (s)": round(float(np.mean(fold_train_times)), 3),
            "inference time per sample mean (ms)": round(float(np.mean(fold_test_times)) * 1000.0, 3),
        })
        print(f"{model_name}: Acc={np.mean(fold_acc):.4f}, Macro F1={np.mean(fold_f1_macro_list):.4f}")

pd.DataFrame(ALL_RESULTS_FOLDS).to_csv(os.path.join(OUT_DIR, "dl_cv_results_folds.csv"), index=False)
pd.DataFrame(ALL_RESULTS_AVG).to_csv(os.path.join(OUT_DIR, "dl_cv_results_avg.csv"), index=False)